# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities via their `@id` values for clarity and reproducibility.

### Dataset Source
The Croissant schema is available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare for exploration using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
ds = mlc.Dataset(croissant_url)

# Display a summary of the dataset via DatasetMetadata attributes
print(f"Dataset name: {ds.metadata.name}\n\nDescription: {ds.metadata.description}\n\nLicense: {ds.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values as defined by the Croissant schema.

In [ ]:
# List all record sets in the dataset with their @id, name, and fields
print("Available record sets:")
record_sets = ds.metadata.record_sets
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    name: {rs.name}")
    field_ids = [f.id for f in rs.fields]
    print(f"    Fields (@id): {field_ids}")
    print()

# As an example, print the first records from the primary record set
if record_sets:
    main_rs_id = record_sets[0].id
    print(f"Example records from '{main_rs_id}':")
    for i, rec in enumerate(ds.records(record_set=main_rs_id)):
        print(rec)
        if i > 2:
            break

## 3. Data Extraction
Extract all records from each record set into separate Pandas DataFrames for further analysis. Reference all record sets by their exact `@id`.

In [ ]:
# Extract all data to DataFrames, with keys being RecordSet @id
dfs = {}
rs_ids = [rs.id for rs in ds.metadata.record_sets]

for rs_id in rs_ids:
    recs = list(ds.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(recs)
    print(f"Loaded DataFrame for RecordSet {rs_id} with shape: {dfs[rs_id].shape}")

# Example: Explore columns for primary record set
main_rs_id = rs_ids[0] if rs_ids else None
if main_rs_id is not None:
    print('Columns in primary record set:', dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Use `@id` references to select numeric and grouping fields. Perform filtering, outlier removal, normalization, and summary aggregation.

In [ ]:
# Choose a numeric field and group field by their @id
# (Adjust these @ids for your actual dataset fields; listed from Section 2 overview)
primary_rs = ds.metadata.record_sets[0]
# Example selection (edit if your dataset differs):
numeric_field_id = None
group_field_id = None

# Identify a likely numeric and group field
for f in primary_rs.fields:
    if getattr(f, 'data_type', None) and ('Float' in str(f.data_type) or 'Integer' in str(f.data_type)) and numeric_field_id is None:
        numeric_field_id = f.id
    if getattr(f, 'data_type', None) and f.data_type == 'schema:Text' and group_field_id is None:
        group_field_id = f.id
    if numeric_field_id and group_field_id:
        break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Proceed only if field detected
df = dfs[primary_rs.id]
if numeric_field_id in df.columns:
    # Remove missing values for analysis
    col = numeric_field_id
    df = df[df[col].notnull()]  # remove NaN
    # Convert to float if not already
    df[col] = pd.to_numeric(df[col], errors='coerce')
    threshold = df[col].mean()  # e.g. filter above mean
    filtered_df = df[df[col] > threshold]
    print(f"Filtered records with {col} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
    print(f"Normalized {col} for filtered records:")
    print(filtered_df[[col, f"{col}_normalized"]].head())

    # Group by group_field (if available)
    if group_field_id is not None and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[col].mean().reset_index()
        print(f"Grouped average of {col} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields (refer to fields by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(12,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook provided an overview and hands-on exploration of the FAIR^2 dataset using the `mlcroissant` library.

- **Metadata and Schema** were loaded via Croissant format and all entity references used their `@id`.
- **Data Overview**: Record sets, fields, and schema structure were inspected.
- **Extraction and EDA**: Data was loaded to DataFrames, filtered, normalized, and visualized using robust, schema-driven field references.
- **Visualization**: Key numeric features and categorical breakdowns were illustrated.

For further analysis, consult the dataset's Croissant schema and documentation to explore more specialized features and to ensure rigorous reproducibility in your ML pipelines.